In [5]:
%pip install matplotlib plotly --quiet

Note: you may need to restart the kernel to use updated packages.


# GDP vs Unemployment — Top 5 Countries Comparison\n\nCompares the **top 5 countries by GDP per capita** (2025, all quarters) against their **unemployment rate**.\n\n- **Bars**: GDP per capita (USD PPP) per quarter — each country has a dedicated color\n- **Lines**: Unemployment rate (%) on the secondary y-axis — same color as the corresponding bars"

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Load merged dataset ──────────────────────────────────────────────────────
df = pd.read_csv("datasets/gdp_unemployment_merged_fixed.csv", sep=";")

# Sort quarters for consistent ordering
quarter_order = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]
df["QUARTER"] = pd.Categorical(df["QUARTER"], categories=quarter_order, ordered=True)
df = df.sort_values(["REF_AREA", "QUARTER"])

# ── Select top-5 countries by mean GDP per capita ────────────────────────────
top5_codes = (
    df.groupby("REF_AREA")["GDP_PER_CAPITA_USD_PPP"]
    .mean()
    .nlargest(5)
    .index.tolist()
)

country_labels = {
    "IRL": "Ireland",
    "LUX": "Luxembourg",
    "NOR": "Norway",
    "USA": "United States",
    "DNK": "Denmark",
}

top5_df = df[df["REF_AREA"].isin(top5_codes)].copy()
top5_df["COUNTRY"] = top5_df["REF_AREA"].map(country_labels)

summary = (
    top5_df.groupby("COUNTRY")["GDP_PER_CAPITA_USD_PPP"]
    .mean()
    .sort_values(ascending=False)
    .rename("Avg GDP per Capita (USD PPP)")
    .map(lambda v: f"${v:,.0f}")
)
print("Top 5 countries by average GDP per capita:")
print(summary.to_string())

Top 5 countries by average GDP per capita:
COUNTRY
Ireland          $126,477
Luxembourg       $119,768
Norway            $74,748
United States     $73,507
Denmark           $69,738


In [6]:
import plotly.graph_objects as go

# ── Color palette — one distinct color per country ───────────────────────────
COLORS = {
    "Ireland":       "#1f77b4",
    "Luxembourg":    "#e07b39",
    "Norway":        "#2ca02c",
    "United States": "#d62728",
    "Denmark":       "#9467bd",
}

quarters = quarter_order
quarter_labels = ["Q1 2025", "Q2 2025", "Q3 2025", "Q4 2025"]

fig = go.Figure()

for code in top5_codes:
    name = country_labels[code]
    color = COLORS[name]
    country_data = (
        top5_df[top5_df["REF_AREA"] == code]
        .set_index("QUARTER")
        .reindex(quarters)
    )

    gdp_vals = country_data["GDP_PER_CAPITA_USD_PPP"].tolist()
    une_vals = country_data["UNE_RATE_PCT"].tolist()

    # GDP bars (primary y-axis)
    fig.add_trace(go.Bar(
        name=name,
        x=quarter_labels,
        y=gdp_vals,
        marker_color=color,
        opacity=0.82,
        yaxis="y1",
        legendgroup=name,
        hovertemplate=(
            f"<b>{name}</b><br>"
            "Quarter: %{x}<br>"
            "GDP per Capita: $%{y:,.0f} USD PPP"
            "<extra></extra>"
        ),
    ))

    # Unemployment lines (secondary y-axis)
    fig.add_trace(go.Scatter(
        name=f"{name} — Unemployment",
        x=quarter_labels,
        y=une_vals,
        mode="lines+markers",
        line=dict(color=color, dash="dash", width=2),
        marker=dict(size=7),
        yaxis="y2",
        legendgroup=name,
        showlegend=True,
        hovertemplate=(
            f"<b>{name}</b><br>"
            "Quarter: %{x}<br>"
            "Unemployment: %{y:.2f}%"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(
        text="Top 5 Countries — GDP per Capita vs Unemployment Rate (2025, by Quarter)",
        font=dict(size=16),
        x=0.5,
    ),
    barmode="group",
    xaxis=dict(title="Quarter", tickfont=dict(size=12)),
    yaxis=dict(
        title="GDP per Capita (USD PPP)",
        tickprefix="$",
        tickformat=",.0f",
        gridcolor="rgba(0,0,0,0.1)",
    ),
    yaxis2=dict(
        title="Unemployment Rate (%)",
        overlaying="y",
        side="right",
        ticksuffix="%",
        tickformat=".1f",
        showgrid=False,
    ),
    legend=dict(
        title="Country  |  Bars = GDP · Dashes = Unemployment",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="lightgray",
        borderwidth=1,
    ),
    hovermode="x unified",
    plot_bgcolor="white",
    width=1050,
    height=560,
)

fig.show()
fig.write_html("unit2-comparison-chart.html")
print("Interactive chart saved as unit2-comparison-chart.html")

Interactive chart saved as unit2-comparison-chart.html
